# 00 Load Data
This notebook loads the Kenya 2022 DHS Individual Recode file, selects the key variables 
for analysis, cleans the data, and saves a cleaned CSV to use for further analysis.

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

In [3]:
df, meta = pyreadstat.read_dta(
    '../Kenya 2022 Individual Recode/KEIR8CFL.DTA')
print(df.shape)
df.head()

(32156, 5925)


,caseid,v000,v001,v002,v003,v004,v005,v006,v007,v008,...,s631l_3,s631l_4,s631l_5,s631l_6,s631m_1,s631m_2,s631m_3,s631m_4,s631m_5,s631m_6
0,1 4 2,KE8,1,4,2,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1 7 2,KE8,1,7,2,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1 10 1,KE8,1,10,1,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 13 2,KE8,1,13,2,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1 20 2,KE8,1,20,2,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Filtering Variables
I filtered out seven variables relevant to my analysis of digital health exclusion.

In [5]:
cols = [
    'v169a', # owns a mobile telephone (0=no, 1=yes)
    'v171a', # use of internet (0=no, 1=yes)
    'v190', # wealth index quintile (1=poorest, 5=richest)
    'v106', # highest educational level (0=none, 1=primary, 2=secondary, 3=higher)
    'v025', # type of place of residence (1=urban, 2=rural)
    'v012', # respondent's current age
    'v005' # sample weight
]

analysis_data = df[cols].copy()
analysis_data.head()

,v169a,v171a,v190,v106,v025,v012,v005
0,1,0,4,0,1,34,1296049
1,1,1,5,2,1,38,1296049
2,1,1,5,1,1,33,1296049
3,1,1,5,2,1,39,1296049
4,1,1,4,1,1,30,1296049


### Dropping Missing Values

In [7]:
before = len(analysis_data)
analysis_data = analysis_data.dropna(subset=['v169a', 'v171a', 'v190', 'v106', 'v025'])
after = len(analysis_data)

print(before)
print(after)


32156
32156


In [8]:
# checking what's happening
for col in cols:
    print(f"{col}: {sorted(df[col].dropna().unique())}")

v169a: [np.int64(0), np.int64(1)]
v171a: [np.int64(0), np.int64(1), np.int64(2)]
v190: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
v106: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
v025: [np.int64(1), np.int64(2)]
v012: [np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49)]
v005: [np.int64(41416), np.int64(50257), np.int64(56442), np.int64(58495), np.int64(63325), np.int64(68641), np.int64(69910), np.int64(71072), np.int64(74770), np.int64(75228), np.int64(76615), np.int64(77662), np.int64(82698), np.int64(84474), np.int64(8478

In [9]:
print(df['v171a'].value_counts())

v171a
0    19328
1    11865
2      963
Name: count, dtype: int64


### Making Independent and Dependent Variables

In [11]:
# dependent variables
analysis_data['phone_owns'] = (analysis_data['v169a'] == 1).astype(int)
analysis_data['internet_use'] = (analysis_data['v171a'] == 1).astype(int)

# v171a values: 0=no, 1=yes, 2=never heard of internet
# recoding 2 as 0 since these women are not internet users
analysis_data['internet_use'] = (analysis_data['v171a'] == 1).astype(int)

# independent variables
analysis_data['wealth_quintile'] = analysis_data['v190'].astype(int)

analysis_data['education'] = analysis_data['v106'].astype(int).map({
    0: 'No education', 1: 'Primary', 2: 'Secondary', 3: 'Higher'})

analysis_data['urban_rural'] = analysis_data['v025'].astype(int).map({
    1: 'Urban', 2: 'Rural'})

analysis_data['maternal_age'] = analysis_data['v012']

analysis_data.head()

,v169a,v171a,v190,v106,v025,v012,v005,phone_owns,internet_use,wealth_quintile,education,urban_rural,maternal_age
0,1,0,4,0,1,34,1296049,1,0,4,No education,Urban,34
1,1,1,5,2,1,38,1296049,1,1,5,Secondary,Urban,38
2,1,1,5,1,1,33,1296049,1,1,5,Primary,Urban,33
3,1,1,5,2,1,39,1296049,1,1,5,Secondary,Urban,39
4,1,1,4,1,1,30,1296049,1,1,4,Primary,Urban,30


### Final Analysis Data

In [13]:
final_cols = [
    'phone_owns',
    'internet_use',
    'wealth_quintile',
    'education',
    'urban_rural',
    'maternal_age'
]

analysis_data = analysis_data[final_cols]
analysis_data.head()

analysis_data.to_csv('../analysis_data.csv', index=False)